# Feature Engineering & Preprocessing Pipeline
This notebook demonstrates a leak-free preprocessing pipeline using a mixed-type Titanic tabular dataset.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
print(df.head())
print('Shape:', df.shape)


In [ ]:
df = df.drop(columns=['PassengerId','Name','Ticket'], errors='ignore')
X = df.drop('Survived', axis=1)
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print('Train:', X_train.shape, 'Test:', X_test.shape)


In [ ]:
numeric_features = X_train.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object','category','bool']).columns.tolist()
print('Numerical:', numeric_features)
print('Categorical:', categorical_features)


In [ ]:
numeric_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer([('num', numeric_pipeline, numeric_features), ('cat', categorical_pipeline, categorical_features)])


In [ ]:
model = Pipeline([('preprocessor', preprocessor), ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('Test Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred))


In [ ]:
rf = model.named_steps['classifier']
fitted_preprocessor = model.named_steps['preprocessor']
feature_names = fitted_preprocessor.get_feature_names_out()
importance = pd.DataFrame({'Feature': feature_names, 'Importance': rf.feature_importances_}).sort_values('Importance', ascending=False)
print(importance.head(15).to_string(index=False))


## Summary
- Train/test split was done before preprocessing.
- Numerical features: imputation + scaling.
- Categorical features: imputation + one-hot encoding.
- ColumnTransformer and Pipeline reduce leakage risk.
- Random Forest feature importance was calculated.